In [16]:
import torch
import torch.nn as nn
import dataclasses
from dataclasses import dataclass

In [ ]:
@dataclass
class Config:
  vocab_size: int = 30522
  n_layers: int = 12
  d_model: int = 768
  n_heads: int = 12
  d_ff: int = 3072 # (4 x d_model)
  dropout: float = 0.1
  pad_id: int = 0
  max_seq_len: int = 128
  n_types: int = 2 # segment A / segment B

In [18]:
class Embeddings(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
    self.pos_embedding = nn.Embedding(config.max_seq_len, config.d_model)
    self.segment_embedding = nn.Embedding(config.n_types, config.d_model)
    self.layer_norm = nn.LayerNorm(config.d_model)
    self.dropout = nn.Dropout(config.dropout)

  def forward(self, x, segment_ids=None):
    B, T = x.size()
    positions = torch.arange(T, device=x.device).unsqueeze(0) # (1, T)

    if segment_ids is None:
      segment_ids = torch.zeros_like(x, dtype=torch.long, device=x.device) # (B, T)

    x = self.token_embedding(x) + self.pos_embedding(positions) + self.segment_embedding(segment_ids) # (B, T, C)
    x = self.layer_norm(x) # (B, T, C)
    x = self.dropout(x) # (B, T, C)
    return x

In [19]:
class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.fc1 = nn.Linear(config.d_model, config.d_ff)
    self.fc2 = nn.Linear(config.d_ff, config.d_model)
    self.gelu = nn.GELU()

  def forward(self, x):
    return self.fc2(self.gelu(self.fc1(x))) # (B, T, d_ff) => (B, T, C)

In [20]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.d_model = config.d_model
    self.n_heads = config.n_heads
    self.head_size = self.d_model // self.n_heads
    self.W_q = nn.Linear(self.d_model, self.d_model)
    self.W_k = nn.Linear(self.d_model, self.d_model)
    self.W_v = nn.Linear(self.d_model, self.d_model)
    self.W_o = nn.Linear(self.d_model, self.d_model)

  def scaled_dot_product(self, Q, K, V, mask=None):
    B, n_heads, T, head_size = Q.size()
    attn_scores = Q @ K.transpose(-1, -2) # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) => (B, n_heads, T, T)
    attn_scores = attn_scores / (head_size ** 0.5) # (B, n_heads, T, T)

    if mask is not None:
      attn_scores = attn_scores.masked_fill(mask==0, float("-inf")) # (B, n_heads, T, T)

    attn_probs = torch.softmax(attn_scores, dim=-1) # (B, n_heads, T, T)
    return attn_probs @ V # (B, n_heads, T, T) @ (B, n_heads, T, head_size) => (B, n_heads, T, head_size)

  def combine_heads(self, x):
    B, n_heads, T, head_size = x.size()
    return x.transpose(1, 2).view(B, T, -1) # (B, T, C)

  def split_heads(self, x):
    B, T, C = x.size()
    return x.view(B, T, self.n_heads, self.head_size).transpose(1, 2) # (B, n_heads, T, C)

  def forward(self, x, mask):
    Q = self.split_heads(self.W_q(x)) # (B, T, C) @ (C, C) => (B, T, C) => (B, n_heads, T, head_size)
    K = self.split_heads(self.W_k(x)) # (B, T, C) @ (C, C) => (B, T, C) => (B, n_heads, T, head_size)
    V = self.split_heads(self.W_v(x)) # (B, T, C) @ (C, C) => (B, T, C) => (B, n_heads, T, head_size)
    attn_output = self.scaled_dot_product(Q, K, V, mask) # (B, n_heads, T, head_size)
    return self.W_o(self.combine_heads(attn_output)) # (B, n_heads, T, head_size) => (B, T, C) @ (C, C) => (B, T, C)

In [21]:
class EncoderLayer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.self_attn = MultiHeadAttention(config)
    self.feed_forward = FeedForward(config)
    self.norm1 = nn.LayerNorm(config.d_model)
    self.norm2 = nn.LayerNorm(config.d_model)
    self.dropout = nn.Dropout(config.dropout)

  def forward(self, x, mask=None):
    attn_output = self.self_attn(x, mask)
    x = self.norm1(x + self.dropout(attn_output))
    ff_output = self.feed_forward(x)
    x = self.norm2(x + self.dropout(ff_output))
    return x

In [ ]:
class BERTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embeddings = Embeddings(config)
        self.encoder_layers = nn.ModuleList(EncoderLayer(config) for _ in range(config.n_layers))
        self.pooler_dense = nn.Linear(config.d_model, config.d_model)
        self.pooler_activation = nn.Tanh()

    def forward(self, input_ids, attn_mask=None, segment_ids=None):
      B, T = input_ids.size()

      if attn_mask is None:
        attn_mask = torch.ones(B, T, device=input_ids.device) # (B, T)

      attn_mask = attn_mask.unsqueeze(1).unsqueeze(2) # (B, 1, 1, T)

      x = self.embeddings(input_ids, segment_ids=segment_ids) # (B, T, C)
      for layer in self.encoder_layers:
          x = layer(x, attn_mask) # (B, T, C)

      cls_token = x[:, 0] # (B, C)
      pooled_output = self.pooler_activation(self.pooler_dense(cls_token)) # (B, C) @ (C, C) => (B, C)
      return x, pooled_output

In [23]:
class MLMHead(nn.Module):
  def __init__(self, config, token_embedding_weight):
    super().__init__()
    self.dense = nn.Linear(config.d_model, config.d_model)
    self.gelu = nn.GELU()
    self.layer_norm = nn.LayerNorm(config.d_model)
    self.decoder = nn.Linear(config.d_model, config.vocab_size, bias=False)
    self.decoder.weight = token_embedding_weight
    self.bias = nn.Parameter(torch.zeros(config.vocab_size))

  def forward(self, x):
    x = self.gelu(self.dense(x)) # (B, T, C) @ (C, C) => (B, T, C)
    x = self.layer_norm(x) # (B, T, C)
    return self.decoder(x) + self.bias # (B, T, C) @ (C, vocab_size) =>  (B, T, vocab_size) + (vocab_size,) => (B, T, vocab_size)

In [24]:
class NSPHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.classifier = nn.Linear(config.d_model, 2) # isNext / NotNext

  def forward(self, x):
    return self.classifier(x) # (B, C) @ (C, 2) => (B, 2)

In [ ]:
class BERTForPreTraining(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BERTModel(config)
        self.mlm_head = MLMHead(config, self.bert.embeddings.token_embedding.weight)
        self.nsp_head = NSPHead(config)

    def forward(self, input_ids, attention_mask=None, segment_ids=None):
        sequence_output, pooled_output = self.bert(input_ids, attention_mask, segment_ids)   
        mlm_logits = self.mlm_head(sequence_output) # (B, T, vocab_size)
        nsp_logits = self.nsp_head(pooled_output) # (B, 2)
        return mlm_logits, nsp_logits